# IBGE Municipalities - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, lower, translate

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.ibge_municipalities"
target_table = f"{catalog}.silver.ibge_municipalities"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

In [0]:
display(bronze_df.limit(10))

In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

In [0]:
for column, dtype in bronze_df.dtypes[:3]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

- ibge_municipality_id is complete and unique. It is the key for this table.

In [0]:
display(bronze_df.select('municipality_name').limit(10))

## Transform to Silver

In [0]:
silver_df = bronze_df.withColumn(
    "municipality_name_normalized",
    translate(
        lower(col("municipality_name")),
        "áàâãäéèêëíìîïóòôõöúùûüç-'",
        "aaaaaeeeeiiiiooooouuuuc  "
    )
)

In [0]:
display(silver_df.select('municipality_name_normalized').limit(20))

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

In [0]:
display(silver_table_df.limit(10))

In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())